# Exp 5 — Signal Alignment: NLI + UMLS Density + UMLS Specificity

End-to-end pipeline that generates biomedical reasoning chains, extracts UMLS concepts,
scores logical entailment, and detects semantic leakage through three independent signals.

| Stage | What it does |
|-------|--------------|
| 0 | Environment setup, API keys, module imports |
| 1 | CoT Generator — 5 LLM chains + 5 prewritten test cases |
| 2 | Concept Extractor — UMLS concept linking for all 10 chains |
| 3 | NLI Entailment + Guard Signals — all chains, all step-pairs |
| 5 | UMLS Signal Computation — density + specificity for all chains |
| 6 | Signal Alignment & Visualisation |

Run cells top-to-bottom in order.
All stages (2–5) operate in full-batch mode across all 10 chains.
Stage 6 visualisations (6a–6d) use chain 1 as a drill-down example;
6e shows the comparison across all 10 chains.


## Stage 0 — Environment Setup

Clones the repo, installs dependencies, configures API keys. Run this first.


In [ ]:
# ── 0a. Clone / pull the repo and configure paths ─────────────────────────
import os, sys
from pathlib import Path

REPO_URL  = 'https://github.com/patmazza25/Bio-Semantic-Drift-Detector'
REPO_DIR  = 'Bio-Semantic-Drift-Detector'

if not Path(REPO_DIR).exists():
    os.system(f'git clone {REPO_URL}')
else:
    os.system(f'git -C {REPO_DIR} pull --quiet')

_cwd = Path(os.getcwd())
if (_cwd / REPO_DIR / 'utils').exists():
    PROJECT_ROOT = str(_cwd / REPO_DIR)
elif (_cwd / 'utils').exists():
    PROJECT_ROOT = str(_cwd)
elif (_cwd.parent / 'utils').exists():
    PROJECT_ROOT = str(_cwd.parent)
else:
    PROJECT_ROOT = str(_cwd / REPO_DIR)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'Python       : {sys.version.split()[0]}')
print(f'Working dir  : {os.getcwd()}')


In [ ]:
# ── 0b. Install dependencies ───────────────────────────────────────────────
!pip install openai numpy pandas scipy scikit-learn matplotlib seaborn requests ipywidgets transformers peft --quiet
print('Dependencies installed.')


In [ ]:
# ── 0c. API key configuration ─────────────────────────────────────────────
import os, importlib.util
os.environ.pop("FORCE_HEURISTIC_NLI", None)  # ensure real NLI model is used
from IPython.display import display, clear_output, HTML


_HAS_WIDGETS = importlib.util.find_spec('ipywidgets') is not None

if _HAS_WIDGETS:
    import ipywidgets as widgets

    # ── OpenRouter key ──────────────────────────────────────────────────────
    _or_box = widgets.Password(
        placeholder='sk-or-v1-…  (get yours free at openrouter.ai)',
        layout=widgets.Layout(width='520px'),
    )
    _or_btn = widgets.Button(
        description='Set Key', button_style='primary',
        icon='check', layout=widgets.Layout(width='110px'),
    )
    _or_out = widgets.Output()

    def _apply_or(_b):
        with _or_out:
            clear_output()
            key = _or_box.value.strip()
            if key:
                os.environ['OPENROUTER_API_KEY'] = key
                print(f'  ✓ OpenRouter key set ({len(key)} chars)')
            else:
                print('  ✗ Paste your OpenRouter key above, then click Set Key.')

    _or_btn.on_click(_apply_or)

    # ── UMLS key ───────────────────────────────────────────────────────────
    _umls_box = widgets.Password(
        placeholder='UMLS API key  (optional — enables density + specificity signals)',
        layout=widgets.Layout(width='520px'),
    )
    _umls_btn = widgets.Button(
        description='Set Key', button_style='info',
        icon='check', layout=widgets.Layout(width='110px'),
    )
    _umls_out = widgets.Output()

    def _apply_umls(_b):
        with _umls_out:
            clear_output()
            key = _umls_box.value.strip()
            if key:
                os.environ['UMLS_API_KEY'] = key
                print(f'  ✓ UMLS key set ({len(key)} chars)')
            else:
                print('  ✗ Paste your UMLS API key above, then click Set Key.')

    _umls_btn.on_click(_apply_umls)

    display(HTML('🔑 OpenRouter API Key (required for CoT generation)'))
    display(widgets.HBox([_or_box, _or_btn]))
    display(_or_out)
    display(HTML('Get a free key at openrouter.ai'))
    display(HTML('🔑 UMLS API Key (optional — enables UMLS density + specificity signals)'))
    display(widgets.HBox([_umls_box, _umls_btn]))
    display(_umls_out)
    display(HTML('Get a key at uts.nlm.nih.gov'))
else:
    os.environ.setdefault('OPENROUTER_API_KEY', '')
    os.environ.setdefault('UMLS_API_KEY', '')
    print('ipywidgets not found — set keys manually:')
    print('  os.environ["OPENROUTER_API_KEY"] = "sk-or-v1-..."')
    print('  os.environ["UMLS_API_KEY"]        = "your-umls-key"')


In [ ]:
# ── 0d. Import all pipeline modules ────────────────────────────────────────
import warnings, json, time, traceback
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings('ignore')

_ok = {}
for mod, sym in [
    ('utils.cot_generator',           'generate'),
    ('utils.concept_extractor_v2',    'extract_concepts'),
    ('utils.hybrid_checker',          'build_entailment_records'),
    ('utils.guards',                  'derive_guards'),
    ('utils.umls_api_linker',         'is_configured'),
    ('utils.umls_density_scorer',     'score_density'),
    ('utils.umls_specificity_scorer', 'score_specificity'),
]:
    try:
        m = __import__(mod, fromlist=[sym])
        _ok[mod] = True
        print(f'  ✓ {mod}')
    except Exception as e:
        _ok[mod] = False
        print(f'  ✗ {mod}: {e}')

from utils.cot_generator           import generate as generate_cot
from utils.hybrid_checker          import build_entailment_records
from utils.guards                  import derive_guards, GuardConfig, lexical_jaccard
from utils.umls_api_linker         import is_configured as umls_configured
from utils.umls_density_scorer     import score_density
from utils.umls_specificity_scorer import score_specificity

# concept_extractor_v2 reads UMLS_API_KEY at call-time (no stale import issue).
# Fall back to the original extractor if v2 is not present.
try:
    from utils.concept_extractor_v2 import extract_concepts
    _extractor_version = 'v2 (standalone REST, call-time key)'
except ImportError:
    from utils.concept_extractor import extract_concepts
    _extractor_version = 'v1 (original)'
    # Patch module-level key so the original extractor sees the env value
    import utils.umls_api_linker as _linker
    _linker.UMLS_API_KEY = os.environ.get('UMLS_API_KEY', '') or _linker.UMLS_API_KEY

GUARD_CFG = GuardConfig()

print()
print(f'  Concept extractor : {_extractor_version}')
print(f'  OpenRouter ready  : {bool(os.environ.get("OPENROUTER_API_KEY"))}')
print(f'  Anthropic ready   : {bool(os.environ.get("ANTHROPIC_API_KEY"))}')
print(f'  UMLS configured   : {umls_configured()}')
print(f'  Heuristic NLI     : {os.environ.get("FORCE_HEURISTIC_NLI", "0") == "1"}')
print()
print('  ↑ Re-run this cell after entering your OpenRouter key above.')
if not umls_configured():
    print()
    print('  ℹ  UMLS not configured — density and specificity signals will show configured=False.')
    print('     NLI signal and visualisations will still work fully.')


In [ ]:
# ── 0e. Pipeline integrity checks ─────────────────────────────────────────
# Verifies all modules are importable, performance fixes are live in the
# running code, and the real NLI model path is properly configured.
import inspect, os as _os

_PASS = 'PASS'
_FAIL = 'FAIL'
_WARN = 'WARN'

checks = []

def _chk(label, ok, note=''):
    checks.append((_PASS if ok else _FAIL, label, note))

def _warn(label, note=''):
    checks.append((_WARN, label, note))

# ── 1. Core module imports ─────────────────────────────────────────────────
for _name, _sym in [
    ('score_density',            'score_density'),
    ('score_specificity',        'score_specificity'),
    ('extract_concepts',         'extract_concepts'),
    ('build_entailment_records', 'build_entailment_records'),
    ('derive_guards',            'derive_guards'),
]:
    _chk(f'import: {_name}', _sym in globals())

# ── 2. UMLS specificity scorer fixes ──────────────────────────────────────
try:
    import utils.umls_specificity_scorer as _spec
    _mcp = getattr(_spec, 'MAX_CONCEPTS_PER_STEP', None)
    _chk('specificity: MAX_CONCEPTS_PER_STEP defined', _mcp is not None,
         f'value={_mcp}' if _mcp is not None else 'missing')
    if _mcp is not None and _mcp > 10:
        _warn('specificity: MAX_CONCEPTS_PER_STEP', f'{_mcp} is high -- recommend <= 5')
except Exception as _e:
    _chk('specificity: MAX_CONCEPTS_PER_STEP', False, str(_e))

try:
    import utils.umls_specificity_scorer as _spec
    _src_atoms = inspect.getsource(_spec._cached_atoms_for_cui)
    _chk('specificity: atoms max_pages=1', 'max_pages=1' in _src_atoms,
         'ok' if 'max_pages=1' in _src_atoms else 'still default (100 pages!)')
except Exception as _e:
    _chk('specificity: atoms max_pages=1', False, str(_e))

try:
    import utils.umls_specificity_scorer as _spec
    _src_anc = inspect.getsource(_spec._cached_ancestor_depth)
    _chk('specificity: ancestors max_pages=1', 'max_pages=1' in _src_anc,
         'ok' if 'max_pages=1' in _src_anc else 'still default (100 pages!)')
except Exception as _e:
    _chk('specificity: ancestors max_pages=1', False, str(_e))

# ---- specificity: atoms_for_cui_direct (fast path) ---------------------
try:
    import utils.umls_specificity_scorer as _spec
    _src_atoms2 = inspect.getsource(_spec._cached_atoms_for_cui)
    _chk('specificity: atoms uses direct endpoint', 'atoms_for_cui_direct' in _src_atoms2,
         'ok' if 'atoms_for_cui_direct' in _src_atoms2 else 'still calling slow atoms_for_cui!')
except Exception as _e:
    _chk('specificity: atoms uses direct endpoint', False, str(_e))

# ── 3. Concept extractor ───────────────────────────────────────────────────
try:
    import utils.concept_extractor_v2 as _cev2
    _chk('concept_extractor_v2 importable', True, 'reads UMLS_API_KEY at call-time')
except ImportError:
    _chk('concept_extractor_v2 importable', False, 'file missing or import error')

# ── 4. NLI model configuration ────────────────────────────────────────────
# ─── 4a. Which NLI mode is active? ──────────────────────────────────────
import os as _os
_fh = _os.getenv('FORCE_HEURISTIC_NLI', '')
_force_heuristic = _fh.strip() in ('1', 'true', 'yes')

if _force_heuristic:
    _chk('NLI: mode', False,
          f'FORCE_HEURISTIC_NLI={_fh!r} -- heuristic override is ON, real model bypassed!')
else:
    try:
        import utils.hybrid_checker as _hc
        if _hc._TRANSFORMERS_OK and _hc._MOD is not None:
            _chk('NLI: mode', True, f'real model loaded: {_hc._MODEL_NAME}')
        else:
            _warn('NLI: mode', 'model not yet loaded -- will load lazily at Stage 3 (normal before Stage 3 runs)')
    except Exception as _nli_e:
        _warn('NLI: mode', f'could not check model state: {_nli_e}')

# 4b. transformers package available
try:
    import transformers as _tr
    _chk('NLI: transformers installed', True, f'v{_tr.__version__}')
except ImportError:
    _chk('NLI: transformers installed', False, 'run setup-0b to install')

# 4c. peft package available (required for LoRA model loading)
try:
    import peft as _peft
    _chk('NLI: peft installed', True, f'v{_peft.__version__}')
except ImportError:
    _chk('NLI: peft installed', False, 'run setup-0b to install')

# 4d. BioNLI model is the primary candidate in hybrid_checker
try:
    import utils.hybrid_checker as _hc
    _src_ensure = inspect.getsource(_hc._ensure_model)
    _has_bionli = 'Bam3752/PubMedBERT-BioNLI-LoRA' in _src_ensure
    _chk('NLI: BioNLI model in candidate list', _has_bionli,
         'ok' if _has_bionli else 'model name missing from _ensure_model')
except Exception as _e:
    _chk('NLI: BioNLI model in candidate list', False, str(_e))

# 4e. PEFT fallback is present in _try_load (handles LoRA adapter format)
try:
    import utils.hybrid_checker as _hc
    _src_load = inspect.getsource(_hc._try_load)
    _has_peft_fb = 'AutoPeftModelForSequenceClassification' in _src_load
    _chk('NLI: PEFT fallback in _try_load', _has_peft_fb,
         'ok' if _has_peft_fb else 'PEFT fallback missing -- LoRA adapters may fail to load')
except Exception as _e:
    _chk('NLI: PEFT fallback in _try_load', False, str(_e))

# ── Print results ──────────────────────────────────────────────────────────
print()
_col_w = 48
_pad   = '-' * (_col_w + 20)
print(_pad)
_hdr = ('Check', 'Status', 'Note')
print(f'  {_hdr[0]:<{_col_w}}  {_hdr[1]}  {_hdr[2]}')
print(_pad)
_fail_count = 0
for _status, _label, _note in checks:
    _note_str = f'  ({_note})' if _note else ''
    print(f'  {_label:<{_col_w}}  {_status:<6}{_note_str}')
    if _status == _FAIL:
        _fail_count += 1
print(_pad)
if _fail_count == 0:
    print('  All checks passed. Pipeline is ready.')
else:
    print(f'  {_fail_count} check(s) FAILED -- review before running stages 2-6.')
print()


## Stage 1 — CoT Generator

Calls an LLM to produce numbered reasoning steps for a biomedical question.
If no API key is set, falls back to 5 generic template steps (provider = local).


In [ ]:
# ── 1a. Configuration ───────────────────────────────────────────────────────
import os
from IPython.display import display, HTML
import importlib.util

_HAS_WIDGETS = importlib.util.find_spec('ipywidgets') is not None

_MODEL_OPTIONS = {
    'Claude Haiku (fast, recommended)':  'anthropic/claude-haiku-4-5',
    'GPT-4o Mini (OpenAI)':              'openai/gpt-4o-mini',
    'Gemini Flash (Google)':             'google/gemini-flash-1.5',
    'Llama 3.3 70B (free tier)':         'meta-llama/llama-3.3-70b-instruct:free',
}

# ── Batch toggle ─────────────────────────────────────────────────────────────
BATCH_TEST = True

if _HAS_WIDGETS:
    import ipywidgets as widgets

    _model_dropdown = widgets.Dropdown(
        options=list(_MODEL_OPTIONS.keys()),
        value='Claude Haiku (fast, recommended)',
        description='Model:',
        layout=widgets.Layout(width='420px'),
    )
    _batch_toggle = widgets.ToggleButton(
        value=True,
        description='Batch test (10 chains)',
        button_style='info',
        icon='list',
        layout=widgets.Layout(width='220px'),
    )
    display(HTML('Model selection — leave as-is if unsure'))
    display(_model_dropdown)
    display(HTML('Batch mode — 5 LLM chains + 5 prewritten test cases'))
    display(_batch_toggle)

    MODEL_ID   = _MODEL_OPTIONS[_model_dropdown.value]
    BATCH_TEST = _batch_toggle.value

    def _on_model_change(change):
        global MODEL_ID
        MODEL_ID = _MODEL_OPTIONS[change['new']]
        print(f'  Model set to: {MODEL_ID}')

    def _on_batch_change(change):
        global BATCH_TEST
        BATCH_TEST = change['new']
        print(f'  Batch mode: {BATCH_TEST}')

    _model_dropdown.observe(_on_model_change, names='value')
    _batch_toggle.observe(_on_batch_change, names='value')
else:
    MODEL_ID   = 'anthropic/claude-haiku-4-5'
    BATCH_TEST = True

PREFER = 'openrouter'
TEST_QUESTION = (
    'Does aspirin reduce the risk of myocardial infarction '
    'in patients with cardiovascular disease?'
)

print(f'Model      : {MODEL_ID}')
print(f'Batch mode : {BATCH_TEST}')
print(f'Question   : {TEST_QUESTION} (single mode only)')


In [ ]:
# ── 1b. Run CoT generation ──────────────────────────────────────────────────
# Reload cot_generator and explicitly patch its module-level API key variables
# so generate_cot picks up the key set in Stage 0c (not the stale import-time value).
import importlib
import utils.cot_generator as _cg_mod
importlib.reload(_cg_mod)

_cg_mod.OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', '')
_cg_mod.OPENROUTER_READY   = bool(_cg_mod.OPENROUTER_API_KEY)
_cg_mod.ANTHROPIC_API_KEY  = os.environ.get('ANTHROPIC_API_KEY', '')
_cg_mod.ANTHROPIC_READY    = bool(_cg_mod.ANTHROPIC_API_KEY)

from utils.cot_generator import generate as generate_cot

# ── Preflight: verify key + model before batch ────────────────────────────
def _openrouter_preflight(api_key, model_id):
    try:
        from openai import OpenAI
        client = OpenAI(api_key=api_key, base_url='https://openrouter.ai/api/v1')
        client.chat.completions.create(
            model=model_id, max_tokens=5,
            messages=[{'role': 'user', 'content': 'hi'}],
            extra_headers={
                'HTTP-Referer': 'https://github.com/biomedical-semantic-leakage',
                'X-Title': 'Biomedical Semantic Leakage Detection',
            },
        )
        return True, None
    except Exception as e:
        return False, str(e)

_key = os.environ.get('OPENROUTER_API_KEY', '')
if _key:
    print(f'Checking OpenRouter key + model ({MODEL_ID})...')
    _pf_ok, _pf_err = _openrouter_preflight(_key, MODEL_ID)
    if _pf_ok:
        print(f'  ✓ OpenRouter ready — model accessible\n')
    else:
        print(f'  ✗ OpenRouter error: {_pf_err}')
        print(f'\n  Common causes:')
        print(f'    • insufficient credits / 402  → switch to a FREE model in Stage 1a')
        print(f'    • invalid api key / 401       → re-paste your key in Stage 0c')
        print(f'    • model not found / 404       → model ID may have changed\n')
else:
    print('  ✗ No API key set — paste your OpenRouter key in Stage 0c and click Set Key.\n')

# ── CoT generation ────────────────────────────────────────────────────────
BATCH_LLM_QUESTIONS = [
    'Does aspirin reduce the risk of myocardial infarction in patients with cardiovascular disease?',
    'How do ACE inhibitors reduce blood pressure in hypertension?',
    'What is the role of TNF-alpha in rheumatoid arthritis joint destruction?',
    'How does chronic kidney disease progress to end-stage renal disease?',
    'What is the mechanism of action of selective serotonin reuptake inhibitors?',
]

LLM_CHAINS = []

if BATCH_TEST:
    print(f'Batch mode — generating {len(BATCH_LLM_QUESTIONS)} LLM chains\n')
    print('=' * 70)

    for qi, q in enumerate(BATCH_LLM_QUESTIONS):
        print(f'\n[{qi+1}/{len(BATCH_LLM_QUESTIONS)}] {q}')
        print('-' * 60)
        t0 = time.time()
        try:
            cot     = generate_cot(q, prefer=PREFER, model=MODEL_ID)
            elapsed = round(time.time() - t0, 2)
            steps   = cot['steps']

            for si, step in enumerate(steps, 1):
                print(f'  Step {si:2d}: {step}')

            print(f'\n  Provider: {cot["provider"]}  |  Model: {cot["model"]}  |  {len(steps)} steps  |  {elapsed}s')

            LLM_CHAINS.append({
                'id':       f'llm_{qi+1}',
                'label':    f'LLM {qi+1} — {q[:55]}',
                'source':   'llm',
                'question': q,
                'expect':   'unknown',
                'steps':    steps,
                'provider': cot['provider'],
                'model':    cot['model'],
            })

            if cot['provider'] == 'local':
                print('  ⚠  provider=local — LLM call failed despite preflight passing.')
                print(f'     OPENROUTER_READY in module: {_cg_mod.OPENROUTER_READY}')
                print(f'     Key in module (len): {len(_cg_mod.OPENROUTER_API_KEY or "")}')

        except Exception as e:
            elapsed = round(time.time() - t0, 2)
            print(f'  ERROR after {elapsed}s: {e}')

        time.sleep(0.3)

    print('\n' + '=' * 70)
    print(f'Generated {len(LLM_CHAINS)}/{len(BATCH_LLM_QUESTIONS)} LLM chains successfully.')

    if LLM_CHAINS:
        STEPS         = LLM_CHAINS[0]['steps']
        PROVIDER      = LLM_CHAINS[0]['provider']
        MODEL_ID_USED = LLM_CHAINS[0]['model']
        print(f'\nSTEPS set to chain 1 for single-chain stages (2–5).')
        print('All stages (2–5) process all 10 chains in full-batch mode.')
    else:
        print('\n⚠  No chains generated — check the preflight error above.')
        STEPS = []
        PROVIDER = 'local'
        MODEL_ID_USED = 'none'

else:
    t0  = time.time()
    cot = generate_cot(TEST_QUESTION, prefer=PREFER, model=MODEL_ID)
    elapsed = round(time.time() - t0, 2)

    STEPS         = cot['steps']
    PROVIDER      = cot['provider']
    MODEL_ID_USED = cot['model']

    print(f'Provider : {PROVIDER}  |  Model : {MODEL_ID_USED}  |  Time : {elapsed}s')
    print(f'Steps    : {len(STEPS)}')
    print()
    for i, step in enumerate(STEPS, 1):
        print(f'  Step {i:2d}: {step}')

    if PROVIDER == 'local':
        print()
        print("⚠  provider='local' — API call failed.")
        print(f'   OPENROUTER_READY in module: {_cg_mod.OPENROUTER_READY}')
        print(f'   Key in module (len): {len(_cg_mod.OPENROUTER_API_KEY or "")}')


In [ ]:
# ── 1c. Validate step quality (chain 1) ─────────────────────────────────────────
# Spot-checks the first LLM chain to confirm the API call succeeded.
checks = {
    'At least 3 steps returned':         len(STEPS) >= 3,
    'All steps non-empty strings':        all(isinstance(s, str) and len(s.strip()) > 0 for s in STEPS),
    'All steps > 15 chars (not trivial)': all(len(s) > 15 for s in STEPS),
    'Real LLM was called (not fallback)': PROVIDER != 'local',
}

all_pass = True
for name, result in checks.items():
    icon = '✓' if result else '✗'
    print(f'  {icon}  {name}')
    if not result:
        all_pass = False

print()
print('Stage 1:', 'PASS ✓' if all_pass else 'WARN — check API key')


In [ ]:
# ── 1d. Define prewritten test cases (batch mode) ───────────────────────────
#
# 5 hand-crafted CoTs covering specific leakage patterns.
# Each has an "expect" label describing what the pipeline SHOULD detect.
#
# CONTROL            → all signals LOW  (no leakage, negative control)
# GRADUAL_LEAKAGE    → UMLS density + specificity drop; NLI stays high (core case)
# CONTRADICTION      → NLI catches abrupt wrong claim (direction flip)
# VAGUENESS_ESCALATION → UMLS signals drop sharply; NLI misses it
# COMPOUNDING_OVERGEN → all signals, gradual progressive drift

PREWRITTEN_COTS = [
    {
        'id':       'control_metformin',
        'label':    'CONTROL — Correct metformin mechanism',
        'source':   'prewritten',
        'question': 'What is the mechanism by which metformin lowers blood glucose?',
        'expect':   'low_risk',
        'steps': [
            'Metformin is a biguanide drug that primarily acts on the liver.',
            'It inhibits mitochondrial complex I, which raises the AMP:ATP ratio in hepatocytes.',
            'The elevated AMP:ATP ratio activates AMP-activated protein kinase (AMPK).',
            'AMPK activation suppresses SREBP-1c and downregulates gluconeogenic enzymes PEPCK and G6Pase.',
            'Reduced gluconeogenesis lowers hepatic glucose output, directly decreasing fasting blood glucose.',
            'Metformin also enhances GLUT4-mediated glucose uptake in skeletal muscle, improving peripheral insulin sensitivity.',
            'Together, these mechanisms lower HbA1c without stimulating insulin release or causing hypoglycemia.',
        ],
    },
    {
        'id':       'gradual_leakage_aspirin',
        'label':    'GRADUAL LEAKAGE — Aspirin overgeneralization',
        'source':   'prewritten',
        'question': 'How does aspirin prevent myocardial infarction?',
        'expect':   'umls_drop',
        'steps': [
            'Aspirin irreversibly acetylates and inhibits cyclooxygenase-1 (COX-1) in platelets.',
            'COX-1 inhibition blocks the conversion of arachidonic acid to thromboxane A2.',
            'Reduced thromboxane A2 impairs platelet activation and aggregation at sites of vascular injury.',
            'Decreased platelet aggregation reduces the likelihood of occlusive arterial thrombus formation.',
            'This antiplatelet effect broadly reduces the tendency for blood to clot in vessels.',
            'Therefore aspirin provides general protection against clotting-related cardiovascular conditions.',
            'Aspirin can consequently be considered a broad protective agent against most vascular diseases.',
        ],
    },
    {
        'id':       'contradiction_statins',
        'label':    'CONTRADICTION — Statin mechanism with abrupt wrong claim',
        'source':   'prewritten',
        'question': 'Do statins reduce LDL cholesterol?',
        'expect':   'nli_contradiction',
        'steps': [
            'Statins competitively inhibit HMG-CoA reductase, the rate-limiting enzyme in the mevalonate pathway.',
            'Inhibiting HMG-CoA reductase reduces endogenous cholesterol synthesis in hepatocytes.',
            'Lower intracellular cholesterol triggers upregulation of LDL receptors on hepatocyte surfaces.',
            'Increased LDL receptor density enhances clearance of LDL particles from the bloodstream.',
            'However, statins do not significantly lower LDL cholesterol levels in clinical practice.',
            'The modest reduction in hepatic synthesis is quickly compensated by increased intestinal cholesterol absorption.',
        ],
    },
    {
        'id':       'vagueness_escalation_insulin',
        'label':    'VAGUENESS ESCALATION — Insulin resistance molecular → vague',
        'source':   'prewritten',
        'question': 'How does insulin resistance develop in type 2 diabetes?',
        'expect':   'umls_drop_sharp',
        'steps': [
            'Insulin binds to its receptor on skeletal muscle, triggering autophosphorylation of the receptor tyrosine kinase domain.',
            'The activated receptor phosphorylates IRS-1 at specific tyrosine residues, creating docking sites for downstream proteins.',
            'Phosphorylated IRS-1 recruits PI3K, which phosphorylates PIP2 to generate PIP3 at the plasma membrane.',
            'PIP3 recruits PDK1 and Akt, leading to Akt phosphorylation and downstream activation of GLUT4 vesicle translocation.',
            'In type 2 diabetes, this signaling cascade becomes progressively impaired at multiple steps.',
            'The impairment leads to a reduced cellular response to normal circulating insulin levels.',
            'This reduced response results in elevated blood glucose and broader metabolic dysfunction.',
        ],
    },
    {
        'id':       'compounding_overgen_af',
        'label':    'COMPOUNDING OVERGENERALIZATION — AF anticoagulation cascade',
        'source':   'prewritten',
        'question': 'Should patients with atrial fibrillation take anticoagulants?',
        'expect':   'all_signals',
        'steps': [
            'Atrial fibrillation causes disorganized atrial electrical activity and loss of effective atrial contraction.',
            'Blood stasis in the left atrial appendage during AF promotes local thrombus formation.',
            'Thrombi originating in the left atrial appendage can embolize to cerebral arteries, causing ischemic stroke.',
            'Direct oral anticoagulants such as apixaban and rivaroxaban reduce stroke risk in AF patients by approximately 65%.',
            'Since anticoagulation reduces thromboembolic events, all patients with irregular heart rhythms should receive it.',
            'Any patient diagnosed with a cardiac arrhythmia is at elevated thromboembolic risk and warrants anticoagulation therapy.',
            'Anticoagulation is therefore broadly indicated for patients with cardiac disease to prevent adverse vascular events.',
        ],
    },
]

if BATCH_TEST:
    ALL_CHAINS = LLM_CHAINS + PREWRITTEN_COTS
    print(f'Batch mode: {len(LLM_CHAINS)} LLM chains + {len(PREWRITTEN_COTS)} prewritten = {len(ALL_CHAINS)} total\n')
    for c in ALL_CHAINS:
        src = '🤖 LLM' if c['source'] == 'llm' else '📝 Prewritten'
        print(f'  {src}  [{c["id"]}]  expect={c["expect"]}')
        print(f'         {c["label"]}')
else:
    ALL_CHAINS = []
    print('Single mode — prewritten cases available but not processed.')
    print('Enable batch mode in Stage 1a to run all 10 chains.')

# Always initialise so Stage 6e never raises NameError
BATCH_RESULTS = []


## Stage 2 — Concept Extractor

Extracts biomedical surface candidates (n-grams, acronyms) from each step
and links them to UMLS concepts (CUIs) if the UMLS API is configured.

| Cell | What it does |
|------|--------------|
| **2a** | Extract concepts for all 10 chains |
| **2b** | Per-chain concept summary table |


In [ ]:
# ── 2a. Concept extraction — all chains ─────────────────────────────────
import re as _re

def _strip_md(text):
    text = _re.sub(r'^#+\s*', '', text, flags=_re.MULTILINE)
    text = _re.sub(r'\*\*(.+?)\*\*', r'\1', text)
    text = _re.sub(r'\*(.+?)\*', r'\1', text)
    text = _re.sub(r'`(.+?)`', r'\1', text)
    return text.strip()

print(f'Extracting concepts for all {len(ALL_CHAINS)} chains...')
print('=' * 70)

for ci, chain in enumerate(ALL_CHAINS):
    steps_c = [_strip_md(s) for s in chain['steps']]
    chain['steps_clean'] = steps_c

    t0 = time.time()
    try:
        concepts = extract_concepts(steps_c, scispacy_when='never', top_k=5)
        chain['concepts'] = concepts
        valid   = sum(1 for step in concepts for c in step if c.get('valid'))
        elapsed = round(time.time() - t0, 2)
        lbl     = chain['label'][:55]
        n_steps = len(steps_c)
        print(f'  [{ci+1}/{len(ALL_CHAINS)}] {lbl}')
        print(f'         {valid} valid concepts across {n_steps} steps  ({elapsed}s)')
    except Exception as e:
        chain['concepts'] = [[] for _ in steps_c]
        print(f'  [{ci+1}/{len(ALL_CHAINS)}] ERROR: {e}')

print()
print('=' * 70)
n_ready = sum(1 for c in ALL_CHAINS if 'concepts' in c)
print(f'Concepts ready for {n_ready}/{len(ALL_CHAINS)} chains.')

# Set chain-1 variables for Stage 6 drill-down visualisations
STEPS_CLEAN = ALL_CHAINS[0]['steps_clean']
CONCEPTS    = ALL_CHAINS[0]['concepts']
print()
print('STEPS_CLEAN and CONCEPTS set to chain 1 for Stage 6 visualisations.')


In [ ]:
# ── 2b. Per-chain concept summary ──────────────────────────────────────────
rows = []
for chain in ALL_CHAINS:
    concepts = chain.get('concepts', [])
    total    = sum(len(step) for step in concepts)
    valid    = sum(1 for step in concepts for c in step if c.get('valid'))
    lbl      = chain['label']
    rows.append({
        'Chain':  lbl[:50] + '...' if len(lbl) > 50 else lbl,
        'Source': chain['source'],
        'Steps':  len(chain.get('steps_clean', chain['steps'])),
        'Total':  total,
        'Valid':  valid,
    })

df_concepts = pd.DataFrame(rows)
display(df_concepts.to_string(index=False))

if not os.environ.get('UMLS_API_KEY', ''):
    print()
    print('  UMLS not configured — all concept counts are 0. Set UMLS_API_KEY in Stage 0c.')


## Stage 3 — NLI Entailment + Guard Signals

Scores every adjacent step-pair for entailment / neutral / contradiction
across all chains, then derives qualitative guard tags on each pair.

| Cell | What it does |
|------|--------------|
| **3a** | Run NLI + guards for all 10 chains |
| **3b** | NLI probability table — all chains, all pairs |
| **3c** | Guard signal table — all chains, all pairs |

**Guard tags** catch cases where the NLI score alone may be misleading:

| Guard | Fires when |
|-------|------------|
| `lexical_duplicate` | Steps are ≥ 90% the same words — chain is repeating itself |
| `caution_band` | Top two probabilities nearly tied — model is uncertain |
| `direction_conflict` | A→B says entailment but B→A says contradiction |


In [ ]:
# ── 3-pre. NLI model selection ────────────────────────────────────────
# Run this cell BEFORE the warm-up cell. It sets BIO_NLI_MODEL (or
# FORCE_HEURISTIC_NLI) so the warm-up downloads exactly the model you pick.
# Nothing is downloaded here.
import os as _os
import importlib.util as _ilu

NLI_MODEL_REGISTRY = {
    'pubmedbert': {
        'id':    'Bam3752/PubMedBERT-BioNLI-LoRA',
        'label': 'PubMedBERT-BioNLI-LoRA  (biomedical, current default)',
    },
    'deberta': {
        'id':    'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli',
        'label': 'DeBERTa-v3-large  91.2% MNLI  (strong general-purpose alternative)',
    },
    'heuristic': {
        'id':    '',
        'label': 'Heuristic only — no model download (token-overlap fallback)',
    },
}

_HAS_WIDGETS = _ilu.find_spec('ipywidgets') is not None

if _HAS_WIDGETS:
    import ipywidgets as _widgets
    from IPython.display import display, clear_output, HTML

    _opts = [f"{k}  —  {v['label']}" for k, v in NLI_MODEL_REGISTRY.items()]
    _nli_radio = _widgets.RadioButtons(
        options=_opts,
        value=_opts[0],
        description='',
        layout=_widgets.Layout(width='780px'),
    )
    _nli_btn = _widgets.Button(
        description='Use this model', button_style='primary',
        icon='check', layout=_widgets.Layout(width='180px'),
    )
    _nli_out = _widgets.Output()

    def _apply_nli(_b):
        with _nli_out:
            clear_output()
            short_key = _nli_radio.value.split('  —  ')[0].strip()
            entry = NLI_MODEL_REGISTRY[short_key]
            model_id = entry['id']
            # Clear both env vars so they don't conflict
            _os.environ.pop('FORCE_HEURISTIC_NLI', None)
            _os.environ.pop('BIO_NLI_MODEL', None)
            if model_id:
                _os.environ['BIO_NLI_MODEL'] = model_id
                print(f'  ✓ NLI model selected: {model_id}')
            else:
                _os.environ['FORCE_HEURISTIC_NLI'] = '1'
                print('  ✓ Heuristic NLI selected — no model will be downloaded.')

    _nli_btn.on_click(_apply_nli)
    display(HTML('<b>🧠 NLI Model Selection</b> — pick a model and click the button <i>before</i> running the warm-up cell.'))
    display(_widgets.VBox([_nli_radio, _widgets.HBox([_nli_btn]), _nli_out]))

else:
    # Non-widget fallback: print instructions
    print('NLI Model Options (ipywidgets not found — set env var manually):')
    for i, (k, v) in enumerate(NLI_MODEL_REGISTRY.items(), 1):
        print(f'  [{i}] {k:12s}  {v["label"]}')
    print()
    print('Set one of these before running the warm-up cell:')
    print("  os.environ['BIO_NLI_MODEL'] = 'Bam3752/PubMedBERT-BioNLI-LoRA'")
    print("  os.environ['BIO_NLI_MODEL'] = 'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli'")
    print("  os.environ['FORCE_HEURISTIC_NLI'] = '1'   # heuristic only, no download")

# ── Guard signals toggle ─────────────────────────────────────────────
# OFF by default. When ON, each pair runs a second reversed NLI pass and
# derive_guards() flags direction conflicts, caution bands, etc.
GUARDS_ENABLED = False  # default

if _HAS_WIDGETS:
    _guards_toggle = _widgets.ToggleButton(
        value=False,
        description='Guards OFF',
        button_style='warning',
        icon='times',
        layout=_widgets.Layout(width='160px'),
    )

    def _on_guards_toggle(change):
        global GUARDS_ENABLED
        GUARDS_ENABLED = change['new']
        _guards_toggle.description = 'Guards ON' if GUARDS_ENABLED else 'Guards OFF'
        _guards_toggle.button_style = 'success' if GUARDS_ENABLED else 'warning'
        _guards_toggle.icon = 'check' if GUARDS_ENABLED else 'times'

    _guards_toggle.observe(_on_guards_toggle, names='value')
    display(HTML('<b>🛡️ Guard Signals</b> — toggle before running the warm-up cell. OFF = skip reverse NLI pass and derive_guards.'))
    display(_guards_toggle)
else:
    print('Guard signals are OFF by default.')
    print('To enable: set  GUARDS_ENABLED = True  before running the warm-up cell.')


In [ ]:
# ── Warm-up: load real NLI model (falls back to heuristic if unavailable) ─
import os as _os
# FORCE_HEURISTIC_NLI is now controlled by the selection cell above
from utils.hybrid_checker import _ensure_model
import utils.hybrid_checker as _hc_mod
_nli_loaded = _ensure_model()
if _nli_loaded:
    print(f'NLI model loaded: {_hc_mod._MODEL_NAME}')
else:
    print('WARNING: NLI model failed to load -- using token-overlap heuristic')
    print('         Install transformers + peft (setup-0b) and restart kernel.')
print()

# ── 3a. NLI + guard derivation — all chains ───────────────────────────────
print(f'Running NLI + guards for all {len(ALL_CHAINS)} chains...')
print('=' * 70)

for ci, chain in enumerate(ALL_CHAINS):
    lbl      = chain['label'][:55]
    steps_c  = chain.get('steps_clean', [_strip_md(s) for s in chain['steps']])
    concepts = chain.get('concepts', [[] for _ in steps_c])

    t0 = time.time()
    try:
        pairs = build_entailment_records(steps_c, concepts)
        chain['pairs'] = pairs

        if GUARDS_ENABLED:
            guarded = []
            for p in pairs:
                i, j = p['step_pair']
                try:
                    rev = build_entailment_records(
                        [steps_c[j], steps_c[i]],
                        [concepts[j] if j < len(concepts) else [],
                         concepts[i] if i < len(concepts) else []],
                    )
                    rev_probs = rev[0]['probs'] if rev else None
                except Exception:
                    rev_probs = None
                guards = derive_guards(
                    premise       = steps_c[i] if i < len(steps_c) else '',
                    hypothesis    = steps_c[j] if j < len(steps_c) else '',
                    probs         = p['probs'],
                    reverse_probs = rev_probs,
                    config        = GUARD_CFG,
                )
                guarded.append({**p, 'guards': guards, 'reverse_probs': rev_probs})
        else:
            guarded = [{**p, 'guards': [], 'reverse_probs': None} for p in pairs]
        chain['guarded_pairs'] = guarded

        elapsed      = round(time.time() - t0, 2)
        label_counts = Counter(p['final_label'] for p in pairs)
        print()
        print(f'  [{ci+1}/{len(ALL_CHAINS)}] {lbl}')
        if GUARDS_ENABLED:
            guard_counts = Counter(g for p in guarded for g in p['guards'])
            print(f'         pairs={len(pairs)}  labels={dict(label_counts)}  guards={dict(guard_counts) or "none"}  ({elapsed}s)')
        else:
            print(f'         pairs={len(pairs)}  labels={dict(label_counts)}  (guards off)  ({elapsed}s)')
    except Exception as e:
        chain['pairs']         = []
        chain['guarded_pairs'] = []
        print(f'  [{ci+1}/{len(ALL_CHAINS)}] ERROR: {e}')

print()
print('=' * 70)
n_ready = sum(1 for c in ALL_CHAINS if 'pairs' in c)
print(f'NLI + guards done for {n_ready}/{len(ALL_CHAINS)} chains.')

# Set chain-1 variables for Stage 6 drill-down
PAIRS         = ALL_CHAINS[0]['pairs']
GUARDED_PAIRS = ALL_CHAINS[0]['guarded_pairs']
print()
print('PAIRS and GUARDED_PAIRS set to chain 1 for Stage 6 visualisations.')


In [ ]:
# ── 3b. NLI probability table — all chains, all pairs ──────────────────────
rows = []
for chain in ALL_CHAINS:
    cid   = chain['id']
    steps = chain.get('steps_clean', chain['steps'])
    for p in chain.get('pairs', []):
        i, j  = p['step_pair']
        probs = p.get('probs', {})
        prem  = steps[i][:45] + '...' if len(steps[i]) > 45 else steps[i]
        hyp   = steps[j][:45] + '...' if len(steps[j]) > 45 else steps[j]
        rows.append({
            'Chain':      cid,
            'Pair':       f'{i}→{j}',
            'Premise':    prem,
            'Hypothesis': hyp,
            'P(entail)':  round(probs.get('entailment',    0), 3),
            'P(neutral)': round(probs.get('neutral',       0), 3),
            'P(contra)':  round(probs.get('contradiction', 0), 3),
            'Label':      p.get('final_label', '?'),
        })

df_nli = pd.DataFrame(rows)

def _colour_label(val):
    colours = {'contradiction': '#ffcccc', 'entailment': '#ccffcc', 'neutral': '#e8e8e8'}
    return 'background-color: ' + colours.get(val, 'white')

display(
    df_nli.style
          .applymap(_colour_label, subset=['Label'])
          .format({'P(entail)': '{:.3f}', 'P(neutral)': '{:.3f}', 'P(contra)': '{:.3f}'})
)


In [ ]:
# ── 3c. Guard signal table — all chains, all pairs ──────────────────────
if not GUARDS_ENABLED:
    print('Guard signals are OFF. Enable the toggle in the selection cell and re-run Stage 3.')
else:
    rows = []
    for chain in ALL_CHAINS:
        cid = chain['id']
        for p in chain.get('guarded_pairs', []):
            i, j   = p['step_pair']
            probs  = p['probs']
            rprobs = p.get('reverse_probs') or {}
            rev_e  = round(rprobs.get('entailment', 0), 3) if rprobs else '—'
            rows.append({
                'Chain':         cid,
                'Pair':          f'{i}→{j}',
                'Label':         p['final_label'],
                'P(contra) fwd': round(probs.get('contradiction', 0), 3),
                'P(entail) rev': rev_e,
                'Guards':        ', '.join(p['guards']) or 'none',
            })

    if rows:
        df_guards = pd.DataFrame(rows)
        display(df_guards.to_string(index=False))
        all_fired = Counter(
            g for chain in ALL_CHAINS
            for p in chain.get('guarded_pairs', [])
            for g in p['guards']
        )
        print()
        print(f'All guards across all chains: {dict(all_fired) or "none"}')
    else:
        print('No guard data — run Stage 3a first.')


## Stage 5 — UMLS Signal Computation

Runs the UMLS chain-level scorers on the concepts extracted in Stage 2.
No additional UMLS API calls are made here.

**Signals:**

| Signal | What it measures | Output |
|--------|-----------------|--------|
| **Density** | Valid UMLS concept count per word, per step | risk (0–1), slope, leakage onset step |
| **Specificity** | Average hierarchy depth of concepts per step | risk (0–1), depth slope, abstraction leaps |

Both signals require `UMLS_API_KEY`. Without it they return `configured: False` and scores are 0.

**Cells:**

| Cell | What it does |
|------|--------------|
| **5a** | Density + specificity for all 10 chains; assembles `BATCH_RESULTS` for Stage 6e |
| **5b** | Per-chain UMLS signal summary table |


In [ ]:
# -- 5a. UMLS signal computation + BATCH_RESULTS assembly -- all chains ------
# Reload UMLS modules so file edits (max_pages=1, concept cap) take effect
# and reduce request timeouts to avoid /content endpoint hangs.
import os as _os, importlib, time
from concurrent.futures import ThreadPoolExecutor, TimeoutError as _FutTimeout

_os.environ['UMLS_MAX_RETRIES'] = '1'
_os.environ['UMLS_REQ_TIMEOUT'] = '4'

import utils.umls_api_linker as _linker_mod
importlib.reload(_linker_mod)
import utils.umls_specificity_scorer as _spec_mod
importlib.reload(_spec_mod)
_score_spec = _spec_mod.score_specificity

SPEC_TIMEOUT = 90  # seconds per chain for specificity before giving up

print(f'Running UMLS scorers for all {len(ALL_CHAINS)} chains...')
print('=' * 70)

_empty_den  = {'configured': False, 'overall_risk': 0.0, 'per_step': [],
               'slope': None, 'leakage_onset_step': None}
_empty_spec = {'configured': False, 'overall_specificity_score': 0.0, 'per_step': [],
               'depth_slope': None, 'abstraction_leaps': []}

for ci, chain in enumerate(ALL_CHAINS):
    lbl      = chain['label'][:55]
    steps_c  = chain.get('steps_clean', [_strip_md(s) for s in chain['steps']])
    concepts = chain.get('concepts', [[] for _ in steps_c])

    t0 = time.time()

    try:
        density = score_density(concepts, steps=steps_c)
    except Exception as _e:
        density = dict(_empty_den)
        print(f'  [{ci+1}/{len(ALL_CHAINS)}] density ERROR: {_e}')

    timed_out = False
    _ex2 = ThreadPoolExecutor(max_workers=1)
    try:
        _fut = _ex2.submit(_score_spec, concepts)
        try:
            specificity = _fut.result(timeout=SPEC_TIMEOUT)
        except _FutTimeout:
            specificity = dict(_empty_spec)
            specificity['configured'] = True
            timed_out = True
        finally:
            _ex2.shutdown(wait=False)
    except Exception as _e:
        specificity = dict(_empty_spec)
        _ex2.shutdown(wait=False)
        print(f'  [{ci+1}/{len(ALL_CHAINS)}] specificity ERROR: {_e}')

    chain['density']     = density
    chain['specificity'] = specificity
    elapsed  = round(time.time() - t0, 2)
    den_risk = density.get('overall_risk', 0.0)
    spc_risk = specificity.get('overall_specificity_score', 0.0)
    onset    = density.get('leakage_onset_step')
    leaps    = len(specificity.get('abstraction_leaps', []))
    spec_sfx = ' [timeout]' if timed_out else ''
    print()
    print(f'  [{ci+1}/{len(ALL_CHAINS)}] {lbl}')
    print(f'         density={den_risk:.3f}  specificity={spc_risk:.3f}{spec_sfx}  onset={onset}  leaps={leaps}  ({elapsed}s)')

print()
print('=' * 70)
n_ready = sum(1 for c in ALL_CHAINS if 'density' in c)
print(f'UMLS scorers done for {n_ready}/{len(ALL_CHAINS)} chains.')

BATCH_RESULTS = []
for chain in ALL_CHAINS:
    if 'pairs' not in chain or 'density' not in chain:
        continue
    pairs   = chain['pairs']
    den     = chain['density']
    spec    = chain['specificity']
    avg_ent = (
        sum(p['probs'].get('entailment', 0.0) for p in pairs) / len(pairs)
        if pairs else 0.0
    )
    BATCH_RESULTS.append({
        'ok':                True,
        'id':                chain['id'],
        'label':             chain['label'],
        'source':            chain['source'],
        'expect':            chain['expect'],
        'concepts':          chain.get('concepts', []),
        'pairs':             pairs,
        'density':           den,
        'specificity':       spec,
        'nli_chain_risk':    round(1.0 - avg_ent, 3),
        'has_contradiction': any(p['final_label'] == 'contradiction' for p in pairs),
        'density_risk':      den.get('overall_risk', 0.0),
        'specificity_risk':  spec.get('overall_specificity_score', 0.0),
        'leakage_onset_step': den.get('leakage_onset_step'),
    })

print()
n_batch = len(BATCH_RESULTS)
print(f'BATCH_RESULTS assembled: {n_batch} chains ready for Stage 6.')

DENSITY     = ALL_CHAINS[0]['density']
SPECIFICITY = ALL_CHAINS[0]['specificity']
print('DENSITY and SPECIFICITY set to chain 1 for Stage 6 visualisations.')

In [ ]:
# ── 5b. Per-chain UMLS signal summary ────────────────────────────────────────
rows = []
for chain in ALL_CHAINS:
    den  = chain.get('density',     {})
    spec = chain.get('specificity', {})
    lbl  = chain['label']
    n_leaps = len(spec.get('abstraction_leaps', []))
    den_sl  = den.get('slope')
    spc_sl  = spec.get('depth_slope')
    rows.append({
        'Chain':      lbl[:45] + '...' if len(lbl) > 45 else lbl,
        'Source':     chain['source'],
        'Expect':     chain['expect'],
        'Den risk':   round(den.get('overall_risk', 0.0), 3),
        'Den slope':  round(den_sl, 4) if den_sl is not None else '—',
        'Onset':      den.get('leakage_onset_step', '—'),
        'Spec risk':  round(spec.get('overall_specificity_score', 0.0), 3),
        'Dep slope':  round(spc_sl, 4) if spc_sl is not None else '—',
        'Leaps':      n_leaps,
    })

df_umls = pd.DataFrame(rows)

def _colour_risk(val):
    if not isinstance(val, (int, float)): return ''
    if val < 0.3:  return 'background-color: #ccffcc'
    if val < 0.6:  return 'background-color: #ffe4b5'
    return 'background-color: #ffcccc'

def _colour_expect(val):
    colours = {'drift': '#ffcccc', 'clean': '#ccffcc'}
    return 'background-color: ' + colours.get(str(val).lower(), 'white')

def _fmt_slope(val):
    return '—' if val == '—' else f'{val:.3f}'

display(
    df_umls.style
           .applymap(_colour_risk,   subset=['Den risk', 'Spec risk'])
           .applymap(_colour_expect, subset=['Expect'])
           .format({'Den risk': '{:.3f}', 'Spec risk': '{:.3f}',
                    'Den slope': _fmt_slope, 'Dep slope': _fmt_slope})
)

if rows and not ALL_CHAINS[0].get('density', {}).get('configured', False):
    print()
    print('  UMLS not configured — scores are 0. Set UMLS_API_KEY in Stage 0c.')


## Stage 6 — Per-Signal Visualisation

Three focused plots, one per signal type. Each uses its natural axis — no shared 0–1 normalisation.

| Cell | What it shows |
|------|--------------|
| **6a** | NLI — P(contradiction) matrix per step-pair, all chains |
| **6b** | UMLS concept density per step — raw ratio + trend, all chains |
| **6c** | UMLS ontology depth per step — ancestor count + trend, all chains |


In [ ]:
# ── 6a. NLI — P(contradiction) matrix, all chains ────────────────────────────
#
# N×N matrix where cell [i,j] = P(contradiction) for pair i→j.
# Only adjacent pairs are scored; non-adjacent cells are grey (NaN).
import math

ncols = 2
nrows = math.ceil(len(ALL_CHAINS) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.5),
                         squeeze=False, constrained_layout=True)

for ci, chain in enumerate(ALL_CHAINS):
    r, c    = divmod(ci, ncols)
    ax      = axes[r, c]

    steps_c = chain.get('steps_clean', chain['steps'])
    pairs   = chain.get('pairs', [])
    n_steps = len(steps_c)

    mat = np.full((n_steps, n_steps), np.nan)
    for p in pairs:
        pi, pj = p['step_pair']
        if pi < n_steps and pj < n_steps:
            mat[pi, pj] = p['probs'].get('contradiction', 0)

    ax.imshow(mat, vmin=0, vmax=1, cmap='RdYlGn_r', aspect='auto')

    ax.set_xticks(range(n_steps))
    ax.set_yticks(range(n_steps))
    ax.set_xticklabels([f'S{i}' for i in range(n_steps)],
                       rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels([f'S{i}' for i in range(n_steps)], fontsize=7)
    ax.set_xlabel('j (hyp)', fontsize=7)
    ax.set_ylabel('i (prem)', fontsize=7)

    for p in pairs:
        pi, pj = p['step_pair']
        if pi < n_steps and pj < n_steps:
            val = mat[pi, pj]
            lbl = p['final_label'][0].upper()
            ax.text(pj, pi, f'{lbl}\n{val:.2f}',
                    ha='center', va='center', fontsize=7,
                    color='white' if val > 0.6 else 'black')

    ax.set_title(chain['label'][:38], fontsize=8, pad=3)

if len(ALL_CHAINS) % 2:
    axes[nrows - 1, 1].set_visible(False)

fig.suptitle(
    '6a — NLI P(contradiction) per Step-Pair  (E=entailment  N=neutral  C=contradiction | colour 0–1)',
    fontsize=10, y=1.01)
plt.show()


In [ ]:
# ── 6b. UMLS concept density per step, all chains ───────────────────────
#
# Per-step: density = valid_concept_count / word_count  (natural ratio, not normalised).
# Trend line uses np.polyfit on the per-step values; slope pre-computed in density['slope'].
# Negative slope = concept grounding decreasing across the chain (leakage signal).
import math

ncols = 2
nrows = math.ceil(len(ALL_CHAINS) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(12, nrows * 3),
                         squeeze=False, constrained_layout=True)

for ci, chain in enumerate(ALL_CHAINS):
    r, c    = divmod(ci, ncols)
    ax      = axes[r, c]
    density = chain.get('density', {})

    if not density.get('configured'):
        ax.text(0.5, 0.5, 'UMLS not configured', transform=ax.transAxes,
                ha='center', va='center', fontsize=9, color='grey', style='italic')
        ax.set_title(chain['label'][:38], fontsize=8, pad=3)
        continue

    per_step = density.get('per_step', [])
    xs = [row['step_index'] for row in per_step]
    ys = [row['density']    for row in per_step]

    ax.bar(xs, ys, color='#4C72B0', alpha=0.6, width=0.6, label='Density')
    ax.plot(xs, ys, 'o', color='#4C72B0', markersize=4)

    if len(xs) >= 2:
        coeffs = np.polyfit(xs, ys, 1)
        trend_ys = np.polyval(coeffs, xs)
        ax.plot(xs, trend_ys, '--', color='#C44E52', linewidth=1.5, label='Trend')

    slope = density.get('slope')
    if slope is not None:
        ax.text(0.98, 0.97, f'slope={slope:.4f}', transform=ax.transAxes,
                ha='right', va='top', fontsize=7,
                color='#C44E52' if slope < 0 else '#2ca02c')

    onset = density.get('leakage_onset_step')
    if onset is not None:
        ax.axvline(onset, color='red', linestyle=':', linewidth=1.2, alpha=0.7,
                   label=f'Onset S{onset}')

    ax.set_xticks(xs)
    ax.set_xticklabels([f'S{x}' for x in xs], fontsize=7)
    ax.set_ylabel('Concepts / word', fontsize=7)
    ax.set_title(chain['label'][:38], fontsize=8, pad=3)
    ax.legend(fontsize=6, loc='upper left', framealpha=0.6)
    ax.grid(axis='y', alpha=0.3)

if len(ALL_CHAINS) % 2:
    axes[nrows - 1, 1].set_visible(False)

fig.suptitle('6b — UMLS Concept Density per Step  (concepts/word, negative slope = grounding loss)',
             fontsize=10, y=1.01)
plt.show()


In [ ]:
# ── 6c. UMLS ontology depth per step, all chains ────────────────────────
#
# Per-step: avg_depth = average ancestor count for grounded concepts  (natural scale).
# Higher depth = more specific (deeper in ontology); lower = more abstract.
# Negative slope = concepts abstracting over the chain. Orange = abstraction leaps.
import math

ncols = 2
nrows = math.ceil(len(ALL_CHAINS) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(12, nrows * 3),
                         squeeze=False, constrained_layout=True)

for ci, chain in enumerate(ALL_CHAINS):
    r, c        = divmod(ci, ncols)
    ax          = axes[r, c]
    specificity = chain.get('specificity', {})

    if not specificity.get('configured'):
        ax.text(0.5, 0.5, 'UMLS not configured', transform=ax.transAxes,
                ha='center', va='center', fontsize=9, color='grey', style='italic')
        ax.set_title(chain['label'][:38], fontsize=8, pad=3)
        continue

    per_step = specificity.get('per_step', [])
    xs_all = [row['step_index'] for row in per_step]
    ys_all = [row['avg_depth']  for row in per_step]

    # Only plot steps where depth is available
    valid = [(x, y) for x, y in zip(xs_all, ys_all) if y is not None]
    if valid:
        vxs, vys = zip(*valid)
        ax.plot(vxs, vys, '^-', color='#55A868', linewidth=1.8,
                markersize=5, label='Depth')

        if len(vxs) >= 2:
            coeffs = np.polyfit(vxs, vys, 1)
            trend_ys = np.polyval(coeffs, vxs)
            ax.plot(vxs, trend_ys, '--', color='#C44E52', linewidth=1.5, label='Trend')

    slope = specificity.get('depth_slope')
    if slope is not None:
        ax.text(0.98, 0.97, f'slope={slope:.4f}', transform=ax.transAxes,
                ha='right', va='top', fontsize=7,
                color='#C44E52' if slope < 0 else '#2ca02c')

    for leap in specificity.get('abstraction_leaps', []):
        ax.axvspan(leap['from_step'] - 0.3, leap['to_step'] + 0.3,
                   alpha=0.15, color='orange',
                   label=f'Leap {leap["from_step"]}→{leap["to_step"]}')

    if xs_all:
        ax.set_xticks(xs_all)
        ax.set_xticklabels([f'S{x}' for x in xs_all], fontsize=7)
    ax.set_ylabel('Ontology depth (ancestors)', fontsize=7)
    ax.set_title(chain['label'][:38], fontsize=8, pad=3)
    ax.legend(fontsize=6, loc='upper left', framealpha=0.6)
    ax.grid(axis='y', alpha=0.3)

if len(ALL_CHAINS) % 2:
    axes[nrows - 1, 1].set_visible(False)

fig.suptitle('6c — UMLS Ontology Depth per Step  (ancestors, negative slope = abstracting, orange = leap)',
             fontsize=10, y=1.01)
plt.show()
